In [1]:
%matplotlib widget

In [2]:
%load_ext autoreload

In [3]:
import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", True)

In [4]:
import numpy as np
import jax.numpy as jnp

In [14]:
%autoreload
from microscope_calibration.model import Model4DSTEM, Parameters4DSTEM, DescanError
from jaxgym.ray import Ray
from jaxgym.run import solve_model
from jaxgym.coordinates import XYCoordinateSystem, XYVector
import jax_dataclasses as jdc

In [19]:
%autoreload

params = Parameters4DSTEM(
    overfocus=1.,  # m
    scan_pixel_pitch=1.,  # m
    scan_cy=1.,  # px
    scan_cx=0.,  # px
    scan_rotation=0.,  # rad
    camera_length=2.,  # m
    detector_pixel_pitch=0.01,  # m
    detector_cy=0.,  # px
    detector_cx=0.,  # px
    semiconv=23/1000,  # rad
    flip_y=False,
    descan_error = DescanError(pxo_pxi=0., offsxi=23, offpyi=0.1)
)

samples = (
    jnp.array((1., 0., 0., 0., 1.)),
    jnp.array((0., 1., 0., 0., 1.)),
    jnp.array((0., 0., 1., 0., 1.)),
    jnp.array((0., 0., 0., 1., 1.)),
    jnp.array((0., 0., 0., 0., 1.)),
)

def sample(args):
    scan_px_x, scan_px_y, source_dx, source_dy, _one = args

    model = Model4DSTEM(params=params)

    ray = model.make_source_ray(source_dx=source_dx, source_dy=source_dy).ray
    
    result = model.trace(
        scan_px_x=scan_px_x,
        scan_px_y=scan_px_y,
        ray=ray,
    )
    scap = result["specimen"].sampling['scan_px']
    detp = scap = result["detector"].sampling['detector_px']
    return jnp.array((scap.x, scap.y, detp.x, detp.y, _one))

res = [sample(args) for args in samples]

input_coords = [(data[0][0], data[0][1], data[1][0], data[1][1], data[1][4]) for data in zip(samples, res)]
output_coords = [(data[1][2], data[1][3], data[1][4]) for data in zip(samples, res)]


In [20]:
jnp.linalg.lstsq(jnp.array(input_coords), jnp.array(output_coords), rcond=None)

(Array([[-7.57204404e-13, -4.26928284e-14, -2.35922393e-16],
        [ 3.05913246e-12, -7.22010440e-14,  6.24500451e-16],
        [ 1.00000000e+00,  1.31361496e-16, -9.56808417e-18],
        [ 9.03161394e-16,  1.00000000e+00, -7.58941521e-18],
        [-2.58505153e-12, -5.09503125e-13,  1.00000000e+00]],      dtype=float64),
 Array([9.09898674e-24, 5.38318682e-26, 9.36437059e-27], dtype=float64),
 Array(5, dtype=int64),
 Array([1.04246807e+04, 2.69328357e+02, 1.00000000e+00, 7.19671252e-01,
        4.45413639e-02], dtype=float64))